## SRP313712

**paper:** [PMID: 33859306](https://pmc.ncbi.nlm.nih.gov/articles/PMC8050052/) - Transcriptional changes and preservation of bone mass in hibernating black bears, 2021

**date, curator:** 2026-01-30, Sara Carsanaro

**notes**
* added summer active / hibernating status to condition based on SRA
* two adults and two 3 years old bears sampled in each group. sexual maturiy occurs between 3-5 years of age -> did not annotate as mature or immature, instead used parent term. no way to know which animals are 3 years vs adults unfortunately 
* sequencing method not clear from paper, however it does suggest ribo-minus (potentially Illumina Total RNA Prep with Ribo-Zero Plus) in [PMID: 22351243](https://pmc.ncbi.nlm.nih.gov/articles/PMC4480364/) which this paper references in its methods -> not enough evidence to mark as ribo-minus, going with default polyA
* Trabecular bone (illium) cores 80 mm long and 20 mm diameter were cut from the ilium tuber coxae of each animal. Bone and bone marrow were not separated. -> based on this annotating with trabecular bone tissue over illium or part of illium (especially because ilium tuber coxae is not in uberon) - although iliac crest (UBERON:0014437) may also be appropriate

### annotation summary

In [20]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,trabecular bone ilium,UBERON:0002483,trabecular bone tissue,missing child term


In [21]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,adult - 3 year or older,UBERON:0000066,fully formed stage,other


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP313712"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|█████████████████████████████████████████████| 8/8 [00:08<00:00,  1.11s/it]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes
Fetching RunIds and ReadHashes for each library...can take several minutes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX10528707,SRP313712,Illumina HiSeq 4000,SRS8652219,,,,,,trabecular bone ilium,adult - 3 year or older,,,,M,,,9643,,,,,,N05-131,SAMN18642352,,,,,,,,summer active,,,29/01/2026,uncpecified,,N05-131,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160191,C2BA05FC6F3DDF1A497D51228BD8E0A1
1,SRX10528706,SRP313712,Illumina HiSeq 4000,SRS8652218,,,,,,trabecular bone ilium,adult - 3 year or older,,,,M,,,9643,,,,,,N04-113,SAMN18642351,,,,,,,,summer active,,,29/01/2026,uncpecified,,N04-113,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160192,FDF7BC8DAB6D521ADFE133F2041AA326
2,SRX10528705,SRP313712,Illumina HiSeq 4000,SRS8652217,,,,,,trabecular bone ilium,adult - 3 year or older,,,,M,,,9643,,,,,,N03-152,SAMN18642350,,,,,,,,summer active,,,29/01/2026,uncpecified,,N03-152,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160193,A1E6064D4FC1EDDF6667C5FA9ABDC02D
3,SRX10528704,SRP313712,Illumina HiSeq 4000,SRS8652216,,,,,,trabecular bone ilium,adult - 3 year or older,,,,M,,,9643,,,,,,N03-111,SAMN18642349,,,,,,,,summer active,,,29/01/2026,uncpecified,,N03-111,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160194,F17BBAD50A25A2464199D0ECEA3E59E0
4,SRX10528703,SRP313712,Illumina HiSeq 4000,SRS8652215,,,,,,trabecular bone ilium,adult - 3 year or older,,,,M,,,9643,,,,,,N05-034,SAMN18642348,,,,,,,,hibernating,,,29/01/2026,uncpecified,,N05-034,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160195,6C1252160F207F4F732C534749290F60
5,SRX10528702,SRP313712,Illumina HiSeq 4000,SRS8652214,,,,,,trabecular bone ilium,adult - 3 year or older,,,,M,,,9643,,,,,,N05-035,SAMN18642347,,,,,,,,hibernating,,,29/01/2026,uncpecified,,N05-035,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160196,1373D2395878C33848739823AEEA5C14
6,SRX10528701,SRP313712,Illumina HiSeq 4000,SRS8652213,,,,,,trabecular bone ilium,adult - 3 year or older,,,,M,,,9643,,,,,,N04-047,SAMN18642346,,,,,,,,hibernating,,,29/01/2026,uncpecified,,N04-047,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160197,A30DB2D5ADB96916463FE0632F087C51
7,SRX10528700,SRP313712,Illumina HiSeq 4000,SRS8652212,,,,,,trabecular bone ilium,adult - 3 year or older,,,,M,,,9643,,,,,,N03-056,SAMN18642345,,,,,,,,hibernating,,,29/01/2026,uncpecified,,N03-056,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160198,8E40E42B6F75CC3D021D8E6DE902B4C9


#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['trabecular bone ilium']


In [6]:

# all
library.loc[:,'anatId'] = 'UBERON:0002483'
library.loc[:,'anatName'] = 'trabecular bone tissue'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'missing child term'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX10528707,SRP313712,Illumina HiSeq 4000,SRS8652219,UBERON:0002483,trabecular bone tissue,,,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,,M,,,9643,,,,,,N05-131,SAMN18642352,,,,,,,,summer active,,,29/01/2026,uncpecified,,N05-131,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160191,C2BA05FC6F3DDF1A497D51228BD8E0A1
1,SRX10528706,SRP313712,Illumina HiSeq 4000,SRS8652218,UBERON:0002483,trabecular bone tissue,,,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,,M,,,9643,,,,,,N04-113,SAMN18642351,,,,,,,,summer active,,,29/01/2026,uncpecified,,N04-113,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160192,FDF7BC8DAB6D521ADFE133F2041AA326
2,SRX10528705,SRP313712,Illumina HiSeq 4000,SRS8652217,UBERON:0002483,trabecular bone tissue,,,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,,M,,,9643,,,,,,N03-152,SAMN18642350,,,,,,,,summer active,,,29/01/2026,uncpecified,,N03-152,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160193,A1E6064D4FC1EDDF6667C5FA9ABDC02D
3,SRX10528704,SRP313712,Illumina HiSeq 4000,SRS8652216,UBERON:0002483,trabecular bone tissue,,,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,,M,,,9643,,,,,,N03-111,SAMN18642349,,,,,,,,summer active,,,29/01/2026,uncpecified,,N03-111,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160194,F17BBAD50A25A2464199D0ECEA3E59E0
4,SRX10528703,SRP313712,Illumina HiSeq 4000,SRS8652215,UBERON:0002483,trabecular bone tissue,,,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,,M,,,9643,,,,,,N05-034,SAMN18642348,,,,,,,,hibernating,,,29/01/2026,uncpecified,,N05-034,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160195,6C1252160F207F4F732C534749290F60
5,SRX10528702,SRP313712,Illumina HiSeq 4000,SRS8652214,UBERON:0002483,trabecular bone tissue,,,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,,M,,,9643,,,,,,N05-035,SAMN18642347,,,,,,,,hibernating,,,29/01/2026,uncpecified,,N05-035,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160196,1373D2395878C33848739823AEEA5C14
6,SRX10528701,SRP313712,Illumina HiSeq 4000,SRS8652213,UBERON:0002483,trabecular bone tissue,,,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,,M,,,9643,,,,,,N04-047,SAMN18642346,,,,,,,,hibernating,,,29/01/2026,uncpecified,,N04-047,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160197,A30DB2D5ADB96916463FE0632F087C51
7,SRX10528700,SRP313712,Illumina HiSeq 4000,SRS8652212,UBERON:0002483,trabecular bone tissue,,,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,,M,,,9643,,,,,,N03-056,SAMN18642345,,,,,,,,hibernating,,,29/01/2026,uncpecified,,N03-056,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160198,8E40E42B6F75CC3D021D8E6DE902B4C9


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['adult - 3 year or older']


In [8]:
# all
library.loc[:,'stageId'] = 'UBERON:0000066'
library.loc[:,'stageName'] = 'fully formed stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'other'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX10528707,SRP313712,Illumina HiSeq 4000,SRS8652219,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,,,,N05-131,SAMN18642352,,,,,,,,summer active,,,29/01/2026,uncpecified,,N05-131,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160191,C2BA05FC6F3DDF1A497D51228BD8E0A1
1,SRX10528706,SRP313712,Illumina HiSeq 4000,SRS8652218,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,,,,N04-113,SAMN18642351,,,,,,,,summer active,,,29/01/2026,uncpecified,,N04-113,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160192,FDF7BC8DAB6D521ADFE133F2041AA326
2,SRX10528705,SRP313712,Illumina HiSeq 4000,SRS8652217,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,,,,N03-152,SAMN18642350,,,,,,,,summer active,,,29/01/2026,uncpecified,,N03-152,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160193,A1E6064D4FC1EDDF6667C5FA9ABDC02D
3,SRX10528704,SRP313712,Illumina HiSeq 4000,SRS8652216,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,,,,N03-111,SAMN18642349,,,,,,,,summer active,,,29/01/2026,uncpecified,,N03-111,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160194,F17BBAD50A25A2464199D0ECEA3E59E0
4,SRX10528703,SRP313712,Illumina HiSeq 4000,SRS8652215,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,,,,N05-034,SAMN18642348,,,,,,,,hibernating,,,29/01/2026,uncpecified,,N05-034,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160195,6C1252160F207F4F732C534749290F60
5,SRX10528702,SRP313712,Illumina HiSeq 4000,SRS8652214,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,,,,N05-035,SAMN18642347,,,,,,,,hibernating,,,29/01/2026,uncpecified,,N05-035,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160196,1373D2395878C33848739823AEEA5C14
6,SRX10528701,SRP313712,Illumina HiSeq 4000,SRS8652213,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,,,,N04-047,SAMN18642346,,,,,,,,hibernating,,,29/01/2026,uncpecified,,N04-047,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160197,A30DB2D5ADB96916463FE0632F087C51
7,SRX10528700,SRP313712,Illumina HiSeq 4000,SRS8652212,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,,,,N03-056,SAMN18642345,,,,,,,,hibernating,,,29/01/2026,uncpecified,,N03-056,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160198,8E40E42B6F75CC3D021D8E6DE902B4C9


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [9]:
# making these variables because we use them again in the experiment file
#my_protocol = ''
# full_length or 3'
#my_protocolType = ''

#library.loc[:,'protocol'] = my_protocol
#library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX10528707,SRP313712,Illumina HiSeq 4000,SRS8652219,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N05-131,SAMN18642352,,,,,,,,summer active,,,29/01/2026,uncpecified,,N05-131,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160191,C2BA05FC6F3DDF1A497D51228BD8E0A1
1,SRX10528706,SRP313712,Illumina HiSeq 4000,SRS8652218,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N04-113,SAMN18642351,,,,,,,,summer active,,,29/01/2026,uncpecified,,N04-113,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160192,FDF7BC8DAB6D521ADFE133F2041AA326
2,SRX10528705,SRP313712,Illumina HiSeq 4000,SRS8652217,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N03-152,SAMN18642350,,,,,,,,summer active,,,29/01/2026,uncpecified,,N03-152,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160193,A1E6064D4FC1EDDF6667C5FA9ABDC02D
3,SRX10528704,SRP313712,Illumina HiSeq 4000,SRS8652216,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N03-111,SAMN18642349,,,,,,,,summer active,,,29/01/2026,uncpecified,,N03-111,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160194,F17BBAD50A25A2464199D0ECEA3E59E0
4,SRX10528703,SRP313712,Illumina HiSeq 4000,SRS8652215,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N05-034,SAMN18642348,,,,,,,,hibernating,,,29/01/2026,uncpecified,,N05-034,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160195,6C1252160F207F4F732C534749290F60
5,SRX10528702,SRP313712,Illumina HiSeq 4000,SRS8652214,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N05-035,SAMN18642347,,,,,,,,hibernating,,,29/01/2026,uncpecified,,N05-035,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160196,1373D2395878C33848739823AEEA5C14
6,SRX10528701,SRP313712,Illumina HiSeq 4000,SRS8652213,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N04-047,SAMN18642346,,,,,,,,hibernating,,,29/01/2026,uncpecified,,N04-047,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160197,A30DB2D5ADB96916463FE0632F087C51
7,SRX10528700,SRP313712,Illumina HiSeq 4000,SRS8652212,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N03-056,SAMN18642345,,,,,,,,hibernating,,,29/01/2026,uncpecified,,N03-056,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160198,8E40E42B6F75CC3D021D8E6DE902B4C9


#### globin, replicates

In [10]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [11]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-01-30'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX10528707,SRP313712,Illumina HiSeq 4000,SRS8652219,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N05-131,SAMN18642352,,,,,,,,summer active,,SAC,2026-01-30,uncpecified,,N05-131,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160191,C2BA05FC6F3DDF1A497D51228BD8E0A1
1,SRX10528706,SRP313712,Illumina HiSeq 4000,SRS8652218,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N04-113,SAMN18642351,,,,,,,,summer active,,SAC,2026-01-30,uncpecified,,N04-113,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160192,FDF7BC8DAB6D521ADFE133F2041AA326
2,SRX10528705,SRP313712,Illumina HiSeq 4000,SRS8652217,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N03-152,SAMN18642350,,,,,,,,summer active,,SAC,2026-01-30,uncpecified,,N03-152,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160193,A1E6064D4FC1EDDF6667C5FA9ABDC02D
3,SRX10528704,SRP313712,Illumina HiSeq 4000,SRS8652216,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N03-111,SAMN18642349,,,,,,,,summer active,,SAC,2026-01-30,uncpecified,,N03-111,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160194,F17BBAD50A25A2464199D0ECEA3E59E0
4,SRX10528703,SRP313712,Illumina HiSeq 4000,SRS8652215,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N05-034,SAMN18642348,,,,,,,,hibernating,,SAC,2026-01-30,uncpecified,,N05-034,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160195,6C1252160F207F4F732C534749290F60
5,SRX10528702,SRP313712,Illumina HiSeq 4000,SRS8652214,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N05-035,SAMN18642347,,,,,,,,hibernating,,SAC,2026-01-30,uncpecified,,N05-035,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160196,1373D2395878C33848739823AEEA5C14
6,SRX10528701,SRP313712,Illumina HiSeq 4000,SRS8652213,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N04-047,SAMN18642346,,,,,,,,hibernating,,SAC,2026-01-30,uncpecified,,N04-047,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160197,A30DB2D5ADB96916463FE0632F087C51
7,SRX10528700,SRP313712,Illumina HiSeq 4000,SRS8652212,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N03-056,SAMN18642345,,,,,,,,hibernating,,SAC,2026-01-30,uncpecified,,N03-056,,,,adult,,TRANSCRIPTOMIC,unspecified,SRR14160198,8E40E42B6F75CC3D021D8E6DE902B4C9


#### comments

In [12]:
library.loc[:,'comment'] = 'iliac crest (UBERON:0014437) may also be appropriate over trabecular bone tissue. some evidence in methods for ribo-minus but not strong enough to treat as ribo-minus'

#### save complete file with correct columns

In [13]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX10528707,SRP313712,Illumina HiSeq 4000,SRS8652219,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N05-131,SAMN18642352,,,,,,,iliac crest (UBERON:0014437) may also be appropriate over trabecular bone tissue. some evidence in methods for ribo-minus but not strong enough to treat as ribo-minus,summer active,,SAC,2026-01-30
1,SRX10528706,SRP313712,Illumina HiSeq 4000,SRS8652218,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N04-113,SAMN18642351,,,,,,,iliac crest (UBERON:0014437) may also be appropriate over trabecular bone tissue. some evidence in methods for ribo-minus but not strong enough to treat as ribo-minus,summer active,,SAC,2026-01-30
2,SRX10528705,SRP313712,Illumina HiSeq 4000,SRS8652217,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N03-152,SAMN18642350,,,,,,,iliac crest (UBERON:0014437) may also be appropriate over trabecular bone tissue. some evidence in methods for ribo-minus but not strong enough to treat as ribo-minus,summer active,,SAC,2026-01-30
3,SRX10528704,SRP313712,Illumina HiSeq 4000,SRS8652216,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N03-111,SAMN18642349,,,,,,,iliac crest (UBERON:0014437) may also be appropriate over trabecular bone tissue. some evidence in methods for ribo-minus but not strong enough to treat as ribo-minus,summer active,,SAC,2026-01-30
4,SRX10528703,SRP313712,Illumina HiSeq 4000,SRS8652215,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N05-034,SAMN18642348,,,,,,,iliac crest (UBERON:0014437) may also be appropriate over trabecular bone tissue. some evidence in methods for ribo-minus but not strong enough to treat as ribo-minus,hibernating,,SAC,2026-01-30
5,SRX10528702,SRP313712,Illumina HiSeq 4000,SRS8652214,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N05-035,SAMN18642347,,,,,,,iliac crest (UBERON:0014437) may also be appropriate over trabecular bone tissue. some evidence in methods for ribo-minus but not strong enough to treat as ribo-minus,hibernating,,SAC,2026-01-30
6,SRX10528701,SRP313712,Illumina HiSeq 4000,SRS8652213,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N04-047,SAMN18642346,,,,,,,iliac crest (UBERON:0014437) may also be appropriate over trabecular bone tissue. some evidence in methods for ribo-minus but not strong enough to treat as ribo-minus,hibernating,,SAC,2026-01-30
7,SRX10528700,SRP313712,Illumina HiSeq 4000,SRS8652212,UBERON:0002483,trabecular bone tissue,UBERON:0000066,fully formed stage,,trabecular bone ilium,adult - 3 year or older,missing child term,not documented,other,M,,,9643,,,polyA,,,N03-056,SAMN18642345,,,,,,,iliac crest (UBERON:0014437) may also be appropriate over trabecular bone tissue. some evidence in methods for ribo-minus but not strong 

### experiment annotations

In [14]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP313712,Transcriptional changes and preservation of bone mass in hibernating black bears,"Physical inactivity leads to losses of bone mass and strength in most mammalian species. In contrast, hibernating bears show no bone loss over the prolonged periods (4 - 6 months) of immobility during winter, which suggests that they have adaptive mechanisms to preserve bone mass. To identify transcriptional changes that underlie molecular mechanisms preventing disuse osteoporosis, we conducted a large-scale gene expression screening in the trabecular bone and bone marrow, comparing hibernating and summer active bears through sequencing of the transcriptome.",SRA,,,,,,,PRJNA720155,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [15]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

8

In [16]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
#experiment.loc[:,'protocol'] = my_protocol
#experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP313712,Transcriptional changes and preservation of bone mass in hibernating black bears,"Physical inactivity leads to losses of bone mass and strength in most mammalian species. In contrast, hibernating bears show no bone loss over the prolonged periods (4 - 6 months) of immobility during winter, which suggests that they have adaptive mechanisms to preserve bone mass. To identify transcriptional changes that underlie molecular mechanisms preventing disuse osteoporosis, we conducted a large-scale gene expression screening in the trabecular bone and bone marrow, comparing hibernating and summer active bears through sequencing of the transcriptome.",SRA,total,Bgee 1K,8,,,,PRJNA720155,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [17]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '33859306'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC8050052/'
experiment.loc[:,'DOI'] = '10.1038/s41598-021-87785-9'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP313712,Transcriptional changes and preservation of bone mass in hibernating black bears,"Physical inactivity leads to losses of bone mass and strength in most mammalian species. In contrast, hibernating bears show no bone loss over the prolonged periods (4 - 6 months) of immobility during winter, which suggests that they have adaptive mechanisms to preserve bone mass. To identify transcriptional changes that underlie molecular mechanisms preventing disuse osteoporosis, we conducted a large-scale gene expression screening in the trabecular bone and bone marrow, comparing hibernating and summer active bears through sequencing of the transcriptome.",SRA,total,Bgee 1K,8,,,,PRJNA720155,33859306,https://pmc.ncbi.nlm.nih.gov/articles/PMC8050052/,10.1038/s41598-021-87785-9,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [18]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [19]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

#### to add things here

#### check columns match

In [ ]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

#### view files

In [ ]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

In [ ]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

### add annotations to git

In [ ]:
! git pull

In [ ]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git status

In [ ]:
! git add $git_experiment_path $git_library_path

In [ ]:
! git commit -m $commit_message_exp

In [ ]:
! git push

### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push